# 01 · Encrypted analytics basics: Snowflake computes, only you can read

```
 LAPTOP (data owner)                                   SNOWFLAKE (standard warehouse, Anaconda pyfhel 3.4.2)
 ┌──────────────────────────────┐   ciphertexts      ┌───────────────────────────────────────────────┐
 │ keys/  secret keys (local)   │ ─────────────────▶ │ EMPLOYEES_ENC   plaintext dims + BINARY cts    │
 │ encrypt salary, bonus, …     │   public params    │ HE_KEYS         public parameters only         │
 │                              │ ─────────────────▶ │ HE_SUM · HE_ADD · HE_SUB · HE_MUL_PLAIN · …    │
 │ decrypt results + verify     │ ◀───────────────── │ GROUP BY / WHERE on plaintext columns          │
 └──────────────────────────────┘ encrypted results  └───────────────────────────────────────────────┘
```

| Scheme | Columns | Arithmetic |
|---|---|---|
| **BFV** | `salary`, `bonus`, `deductions` (USD) | exact integers; results must stay within ±t/2 ≈ ±8.6·10⁹ |
| **CKKS** | `hourly_rate`, `hours`, `perf_score`, `tenure`, `remote_flag` | approximate reals (~1e-6 relative at depth 1) |

Every number below is computed by Snowflake **on ciphertexts**, returned **encrypted**, decrypted here,
and checked against the plaintext ground truth, which never leaves the laptop.

In [1]:
import numpy as np
import pandas as pd

from he_snowflake import data
from he_snowflake.config import get_session, load_config
from he_snowflake.crypto import bfv_range, decrypt, noise_budget
from he_snowflake.deploy import deploy
from he_snowflake.keys import load_all
from he_snowflake.load import load
from he_snowflake.verify import Checks, run_encrypted

pd.set_option("display.float_format", "{:,.4f}".format)
cfg = load_config()
session = get_session(cfg, create_schema=True)
keys = load_all(cfg.key_dir, max_packed_values=cfg.n_employees)
BFV, CKKS = keys["bfv"], keys["ckks"]
emp = data.employees(cfg.n_employees, cfg.seed)      # plaintext ground truth (local only)
checks = Checks()
print(session.sql("SELECT CURRENT_ROLE() AS role, CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS schema").collect()[0])

Row(ROLE='SYSADMIN', DB='HE_DEMO', SCHEMA='PYFHEL')


## 1. Deploy functions and load encrypted data
`deploy` creates the tables and the homomorphic functions; `load` encrypts every sensitive value locally and
uploads only ciphertexts plus public parameters. Set `LOAD = False` to reuse what is already in Snowflake.

In [2]:
LOAD = True
if LOAD:
    deploy(session)
    load(session, cfg, keys)
emp.head(5)

{
  "encrypt_s": 9.5,
  "employee_ciphertext_mb": 182.3,
  "upload_s": 121.9,
  "rows": {
    "HE_KEYS": 4,
    "EMPLOYEES_ENC": 100,
    "REVENUE_MONTHLY_ENC": 36,
    "PACKED_COLUMNS": 4,
    "FX_RATES": 3
  }
}


,emp_id,department,region,hire_year,fte_pct,salary_usd,bonus_usd,deductions_usd,hourly_rate,hours_logged,perf_score,tenure_years,remote_flag
0,1,Marketing,AMER,2017,100,102963,11447,11176,145.5000,46.9300,4.1100,9.5000,1
1,2,Sales,APJ,2023,100,179943,28690,19657,175.6300,25.8600,3.7200,3.3000,0
2,3,Support,EMEA,2012,100,82228,15094,8672,102.6100,54.5000,3.0500,14.0000,1
3,4,Finance,AMER,2024,100,152973,23331,22800,47.3000,39.6900,2.0900,2.5000,1
4,5,Engineering,APJ,2019,100,174890,33884,18658,27.1600,16.3000,2.2400,8.0000,0


## 2. What Snowflake actually holds

In [3]:
session.sql("""
SELECT emp_id, department, region,
       LENGTH(salary_ct)                            AS salary_ct_bytes,
       HEX_ENCODE(SUBSTR(salary_ct, 4097, 12))      AS salary_ct_sample,
       LENGTH(hourly_rate_ct)                       AS rate_ct_bytes,
       HEX_ENCODE(SUBSTR(hourly_rate_ct, 4097, 12)) AS rate_ct_sample
FROM EMPLOYEES_ENC ORDER BY emp_id LIMIT 5""").to_pandas()

,EMP_ID,DEPARTMENT,REGION,SALARY_CT_BYTES,SALARY_CT_SAMPLE,RATE_CT_BYTES,RATE_CT_SAMPLE
0,1,Marketing,AMER,88524,35C7E405A5CD10E655376AE0,312415,A4ACC959010E3366A2B1A24E
1,2,Sales,APJ,88480,B9DDB7B495499FD123C6C3A4,312544,E3264FF380974A921E389A6B
2,3,Support,EMEA,88579,ED3FA9A8917DC5C696A64F6C,311698,7BE78969C607048875C30CDC
3,4,Finance,AMER,88532,51BF9F6C1FE3571F8B901B96,311340,EE7D1591E971FF98CA505807
4,5,Engineering,APJ,88503,DDC2A7867A9DFBBAE4E560F1,312634,04060394C0E414237D36EAA4


In [4]:
session.sql("""
SELECT key_id, scheme, mode, LENGTH(context) AS public_context_bytes,
       COALESCE(LENGTH(rotation_keys), 0) AS public_rotation_key_bytes
FROM HE_KEYS ORDER BY key_id""").to_pandas()

,KEY_ID,SCHEME,MODE,PUBLIC_CONTEXT_BYTES,PUBLIC_ROTATION_KEY_BYTES
0,bfv-04fca773bf98,bfv,row,96,0
1,bfv_packed-94694ee89b04,bfv,packed,96,1931864
2,ckks-63ed7de84d25,ckks,row,122,0
3,ckks_packed-e4c986944ca4,ckks,packed,122,9220844


Row-mode keys upload **only the ~100-byte public context** (ring size, moduli, scale). There is no secret key,
no public key and no evaluation key in Snowflake: it can add and multiply ciphertexts, but it can neither
decrypt nor even encrypt. Packed mode (notebook 03) additionally uploads public rotation keys.

## 3. SUM, COUNT, AVG over whole encrypted columns

In [5]:
SQL = """
SELECT COUNT(*) AS n,
       HE_SUM(k.context, e.salary_ct)     AS salary_sum,
       HE_SUM(k.context, e.bonus_ct)      AS bonus_sum,
       HE_SUM(k.context, e.deductions_ct) AS deductions_sum
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.bfv_key_id"""
raw = session.sql(SQL).collect()[0]
print(f"encrypted total salary returned by Snowflake: {len(raw['SALARY_SUM']):,} bytes, "
      f"noise budget left {noise_budget(BFV, raw['SALARY_SUM'])} bits")
r, secs = run_encrypted(session, SQL, {"SALARY_SUM": BFV, "BONUS_SUM": BFV, "DEDUCTIONS_SUM": BFV})
r = r.iloc[0]
print(f"query {secs:.1f}s")
for col, plain in (("SALARY_SUM", "salary_usd"), ("BONUS_SUM", "bonus_usd"), ("DEDUCTIONS_SUM", "deductions_usd")):
    checks.check(f"SUM({plain})", r[col], emp[plain].sum())
checks.check("AVG(salary_usd)", r.SALARY_SUM / r.N, emp.salary_usd.mean(), rel_tol=1e-12)

encrypted total salary returned by Snowflake: 88,521 bytes, noise budget left 27 bits


query 7.5s
OK       SUM(salary_usd): decrypted=13,996,222.000000 plaintext=13,996,222.000000 abs_err=0.00e+00
OK       SUM(bonus_usd): decrypted=1,687,758.000000 plaintext=1,687,758.000000 abs_err=0.00e+00
OK       SUM(deductions_usd): decrypted=1,659,307.000000 plaintext=1,659,307.000000 abs_err=0.00e+00
OK       AVG(salary_usd): decrypted=139,962.220000 plaintext=139,962.220000 abs_err=0.00e+00


## 4. GROUP BY and ROLLUP on plaintext dimensions

In [6]:
SQL = """
SELECT COALESCE(e.department, '<ALL>') AS department, COUNT(*) AS headcount,
       HE_SUM(k.context, e.salary_ct) AS payroll
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.bfv_key_id
GROUP BY ROLLUP (e.department) ORDER BY department"""
dept, secs = run_encrypted(session, SQL, {"PAYROLL": BFV})
truth = emp.groupby("department").salary_usd.agg(["count", "sum"])
truth.loc["<ALL>"] = [len(emp), emp.salary_usd.sum()]
dept = dept.set_index("DEPARTMENT")
dept["payroll (plaintext)"] = truth["sum"]
dept["avg salary"] = dept.PAYROLL / dept.HEADCOUNT
dept["share of total"] = dept.PAYROLL / dept.loc["<ALL>", "PAYROLL"]
checks.frame("payroll by department (ROLLUP)", dept, "PAYROLL", "payroll (plaintext)")
print(f"query {secs:.1f}s")
dept

OK       payroll by department (ROLLUP) [<ALL>]: decrypted=13,996,222.000000 plaintext=13,996,222.000000 abs_err=0.00e+00
OK       payroll by department (ROLLUP) [Engineering]: decrypted=5,741,307.000000 plaintext=5,741,307.000000 abs_err=0.00e+00
OK       payroll by department (ROLLUP) [Finance]: decrypted=2,566,543.000000 plaintext=2,566,543.000000 abs_err=0.00e+00
OK       payroll by department (ROLLUP) [Marketing]: decrypted=2,098,029.000000 plaintext=2,098,029.000000 abs_err=0.00e+00
OK       payroll by department (ROLLUP) [Sales]: decrypted=2,962,582.000000 plaintext=2,962,582.000000 abs_err=0.00e+00
OK       payroll by department (ROLLUP) [Support]: decrypted=627,761.000000 plaintext=627,761.000000 abs_err=0.00e+00
query 5.6s


,HEADCOUNT,PAYROLL,payroll (plaintext),avg salary,share of total
DEPARTMENT,,,,,
<ALL>,100,13996222,13996222,"139,962.2200",1.0000
Engineering,33,5741307,5741307,"173,979.0000",0.4102
Finance,19,2566543,2566543,"135,081.2105",0.1834
Marketing,19,2098029,2098029,"110,422.5789",0.1499
Sales,21,2962582,2962582,"141,075.3333",0.2117
Support,8,627761,627761,"78,470.1250",0.0449


## 5. Plaintext filters: `WHERE` works on non-encrypted columns

In [7]:
SQL = """
SELECT COUNT(*) AS n, HE_SUM(k.context, e.salary_ct) AS payroll
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.bfv_key_id
WHERE e.region = 'EMEA' AND e.hire_year >= 2020"""
r, _ = run_encrypted(session, SQL, {"PAYROLL": BFV})
sub = emp[(emp.region == "EMEA") & (emp.hire_year >= 2020)]
checks.check("SUM(salary) WHERE region=EMEA AND hire_year>=2020", r.PAYROLL[0], sub.salary_usd.sum())
checks.check("COUNT(*) same filter", r.N[0], len(sub))

OK       SUM(salary) WHERE region=EMEA AND hire_year>=2020: decrypted=2,376,556.000000 plaintext=2,376,556.000000 abs_err=0.00e+00
OK       COUNT(*) same filter: decrypted=16.000000 plaintext=16.000000 abs_err=0.00e+00


## 6. Encrypted ± encrypted: gross pay and net pay
`net = salary + bonus − deductions`, row by row on ciphertexts, then summed per department.

In [8]:
SQL = """
SELECT e.department,
       HE_SUM(k.context, HE_ADD(k.context, e.salary_ct, e.bonus_ct)) AS gross,
       HE_SUM(k.context, HE_SUB(k.context, HE_ADD(k.context, e.salary_ct, e.bonus_ct), e.deductions_ct)) AS net
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.bfv_key_id
GROUP BY e.department ORDER BY e.department"""
pay, secs = run_encrypted(session, SQL, {"GROSS": BFV, "NET": BFV})
pay = pay.set_index("DEPARTMENT")
pay["gross (plaintext)"] = (emp.salary_usd + emp.bonus_usd).groupby(emp.department).sum()
pay["net (plaintext)"] = (emp.salary_usd + emp.bonus_usd - emp.deductions_usd).groupby(emp.department).sum()
checks.frame("gross pay", pay, "GROSS", "gross (plaintext)")
checks.frame("net pay", pay, "NET", "net (plaintext)")
pay

OK       gross pay [Engineering]: decrypted=6,423,148.000000 plaintext=6,423,148.000000 abs_err=0.00e+00
OK       gross pay [Finance]: decrypted=2,870,461.000000 plaintext=2,870,461.000000 abs_err=0.00e+00
OK       gross pay [Marketing]: decrypted=2,353,483.000000 plaintext=2,353,483.000000 abs_err=0.00e+00
OK       gross pay [Sales]: decrypted=3,309,106.000000 plaintext=3,309,106.000000 abs_err=0.00e+00
OK       gross pay [Support]: decrypted=727,782.000000 plaintext=727,782.000000 abs_err=0.00e+00
OK       net pay [Engineering]: decrypted=5,775,987.000000 plaintext=5,775,987.000000 abs_err=0.00e+00
OK       net pay [Finance]: decrypted=2,543,724.000000 plaintext=2,543,724.000000 abs_err=0.00e+00
OK       net pay [Marketing]: decrypted=2,089,500.000000 plaintext=2,089,500.000000 abs_err=0.00e+00
OK       net pay [Sales]: decrypted=2,966,032.000000 plaintext=2,966,032.000000 abs_err=0.00e+00
OK       net pay [Support]: decrypted=649,430.000000 plaintext=649,430.000000 abs_err=0.00e+00


,GROSS,NET,gross (plaintext),net (plaintext)
DEPARTMENT,,,,
Engineering,6423148,5775987,6423148,5775987
Finance,2870461,2543724,2870461,2543724
Marketing,2353483,2089500,2353483,2089500
Sales,3309106,2966032,3309106,2966032
Support,727782,649430,727782,649430


## 7. Constant arithmetic: deductions, sign flips, raises, linear combinations
BFV is exact integer arithmetic, so percentages are applied in hundredths and divided after decryption:
* flat deduction: `salary − 1,200` (health premium)
* negation: `−deductions` (book deductions as a credit)
* 3% raise + $1,000 bonus: `salary·103 + 1000·100` → `/100`
* employer cost: `1.08·salary + bonus + 1,200` → `108·salary + 100·bonus + 120,000` → `/100`

In [9]:
SQL = """
SELECT HE_SUM(k.context, HE_ADD_PLAIN(k.context, e.salary_ct, -1200))                     AS after_premium,
       HE_SUM(k.context, HE_NEG(k.context, e.deductions_ct))                              AS neg_deductions,
       HE_SUM(k.context, HE_ADD_PLAIN(k.context, HE_MUL_PLAIN(k.context, e.salary_ct, 103), 100000)) AS raise_x100,
       HE_SUM(k.context, HE_ADD_PLAIN(k.context,
                         HE_ADD(k.context, HE_MUL_PLAIN(k.context, e.salary_ct, 108),
                                           HE_MUL_PLAIN(k.context, e.bonus_ct, 100)), 120000)) AS employer_cost_x100
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.bfv_key_id"""
r, secs = run_encrypted(session, SQL, {c: BFV for c in ("AFTER_PREMIUM", "NEG_DEDUCTIONS", "RAISE_X100", "EMPLOYER_COST_X100")})
r = r.iloc[0]
checks.check("SUM(salary - 1200)", r.AFTER_PREMIUM, (emp.salary_usd - 1200).sum())
checks.check("SUM(-deductions)", r.NEG_DEDUCTIONS, -emp.deductions_usd.sum())
checks.check("SUM(salary*1.03 + 1000)", r.RAISE_X100 / 100, (emp.salary_usd * 1.03 + 1000).sum(), abs_tol=1e-6)
checks.check("SUM(1.08*salary + bonus + 1200)", r.EMPLOYER_COST_X100 / 100,
             (1.08 * emp.salary_usd + emp.bonus_usd + 1200).sum(), abs_tol=1e-6)
print(f"query {secs:.1f}s | largest intermediate {r.EMPLOYER_COST_X100:,} vs BFV limit ±{bfv_range(BFV):,}")

OK       SUM(salary - 1200): decrypted=13,876,222.000000 plaintext=13,876,222.000000 abs_err=0.00e+00
OK       SUM(-deductions): decrypted=-1,659,307.000000 plaintext=-1,659,307.000000 abs_err=0.00e+00
OK       SUM(salary*1.03 + 1000): decrypted=14,516,108.660000 plaintext=14,516,108.660000 abs_err=0.00e+00
OK       SUM(1.08*salary + bonus + 1200): decrypted=16,923,677.760000 plaintext=16,923,677.760000 abs_err=0.00e+00
query 12.4s | largest intermediate 1,692,367,776 vs BFV limit ±8,589,877,248


## 8. Weighted average with plaintext weights (FTE-weighted salary)
`Σ(fte_pct · salary) / Σ fte_pct`: the weights are plaintext, the salaries stay encrypted.

In [10]:
SQL = """
SELECT SUM(e.fte_pct) AS w_sum,
       HE_SUM(k.context, HE_MUL_PLAIN(k.context, e.salary_ct, e.fte_pct)) AS weighted
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.bfv_key_id"""
r, _ = run_encrypted(session, SQL, {"WEIGHTED": BFV})
checks.check("FTE-weighted avg salary", r.WEIGHTED[0] / r.W_SUM[0],
             (emp.fte_pct * emp.salary_usd).sum() / emp.fte_pct.sum(), rel_tol=1e-12)

OK       FTE-weighted avg salary: decrypted=139,904.868910 plaintext=139,904.868910 abs_err=0.00e+00


## 9. CKKS decimals: averages, uplift, and currency conversion via a plaintext join

In [11]:
SQL = """
SELECT e.region, COUNT(*) AS n, MAX(f.usd_to_local) AS fx,
       HE_SUM(k.context, e.hourly_rate_ct)                                  AS rate_sum,
       HE_SUM(k.context, HE_MUL_PLAIN(k.context, e.hourly_rate_ct, 1.05))   AS uplift_sum,
       HE_SUM(k.context, HE_MUL_PLAIN(k.context, e.hourly_rate_ct, f.usd_to_local)) AS local_sum
FROM EMPLOYEES_ENC e
JOIN HE_KEYS k  ON k.key_id = e.ckks_key_id
JOIN FX_RATES f ON f.region = e.region
GROUP BY e.region ORDER BY e.region"""
fx, secs = run_encrypted(session, SQL, {"RATE_SUM": CKKS, "UPLIFT_SUM": CKKS, "LOCAL_SUM": CKKS})
fx = fx.set_index("REGION")
g = emp.groupby("region").hourly_rate
fx["avg rate USD"] = fx.RATE_SUM / fx.N
fx["avg rate +5%"] = fx.UPLIFT_SUM / fx.N
fx["avg rate local ccy"] = fx.LOCAL_SUM / fx.N
fx["avg rate local (plaintext)"] = g.mean() * pd.Series(data.FX_RATES)
checks.frame("avg hourly rate", fx.assign(t=g.mean()), "avg rate USD", "t", rel_tol=1e-6)
checks.frame("avg hourly rate +5%", fx.assign(t=g.mean() * 1.05), "avg rate +5%", "t", rel_tol=1e-6)
checks.frame("avg hourly rate in local currency", fx, "avg rate local ccy", "avg rate local (plaintext)", rel_tol=1e-6)
fx.drop(columns=["RATE_SUM", "UPLIFT_SUM", "LOCAL_SUM"])

OK       avg hourly rate [AMER]: decrypted=94.581064 plaintext=94.581064 abs_err=1.46e-09
OK       avg hourly rate [APJ]: decrypted=106.839444 plaintext=106.839444 abs_err=1.04e-08
OK       avg hourly rate [EMEA]: decrypted=98.599714 plaintext=98.599714 abs_err=9.30e-10
OK       avg hourly rate +5% [AMER]: decrypted=99.310117 plaintext=99.310117 abs_err=1.73e-09
OK       avg hourly rate +5% [APJ]: decrypted=112.181417 plaintext=112.181417 abs_err=3.74e-09
OK       avg hourly rate +5% [EMEA]: decrypted=103.529700 plaintext=103.529700 abs_err=1.85e-09
OK       avg hourly rate in local currency [AMER]: decrypted=94.581064 plaintext=94.581064 abs_err=2.18e-11
OK       avg hourly rate in local currency [APJ]: decrypted=16,175.491890 plaintext=16,175.491889 abs_err=1.57e-06
OK       avg hourly rate in local currency [EMEA]: decrypted=90.711737 plaintext=90.711737 abs_err=3.51e-10


,N,FX,avg rate USD,avg rate +5%,avg rate local ccy,avg rate local (plaintext)
REGION,,,,,,
AMER,47,1.0000,94.5811,99.3101,94.5811,94.5811
APJ,18,151.4000,106.8394,112.1814,"16,175.4919","16,175.4919"
EMEA,35,0.9200,98.5997,103.5297,90.7117,90.7117


## 10. Limits: BFV wraps around beyond ±t/2
Multiplying total payroll by 1,000 exceeds t/2. Snowflake happily computes it; the decrypted value is
the true result modulo t. Keep BFV results inside the range, or move the computation to CKKS.

In [12]:
SQL = """
SELECT HE_MUL_PLAIN(MAX(k.context), HE_SUM(k.context, e.salary_ct), 1000) AS times_1000
FROM EMPLOYEES_ENC e JOIN HE_KEYS k ON k.key_id = e.bfv_key_id"""
r, _ = run_encrypted(session, SQL, {"TIMES_1000": BFV})
true = int(emp.salary_usd.sum()) * 1000
t = BFV.he.t
print(f"true {true:,} | decrypted {r.TIMES_1000[0]:,} | true mod t (signed) {((true + t // 2) % t) - t // 2:,} | limit ±{t // 2:,}")

true 13,996,222,000 | decrypted -3,183,532,497 | true mod t (signed) -3,183,532,497 | limit ±8,589,877,248


In [13]:
checks.summary()

36/36 checks passed


,check,decrypted,plaintext,abs_err,rel_err,ok
0,SUM(salary_usd),"13,996,222.0000","13,996,222.0000",0.0000,0.0000,True
1,SUM(bonus_usd),"1,687,758.0000","1,687,758.0000",0.0000,0.0000,True
2,SUM(deductions_usd),"1,659,307.0000","1,659,307.0000",0.0000,0.0000,True
3,AVG(salary_usd),"139,962.2200","139,962.2200",0.0000,0.0000,True
4,payroll by department (ROLLUP) [<ALL>],"13,996,222.0000","13,996,222.0000",0.0000,0.0000,True
5,payroll by department (ROLLUP) [Engineering],"5,741,307.0000","5,741,307.0000",0.0000,0.0000,True
6,payroll by department (ROLLUP) [Finance],"2,566,543.0000","2,566,543.0000",0.0000,0.0000,True
7,payroll by department (ROLLUP) [Marketing],"2,098,029.0000","2,098,029.0000",0.0000,0.0000,True
8,payroll by department (ROLLUP) [Sales],"2,962,582.0000","2,962,582.0000",0.0000,0.0000,True
9,payroll by department (ROLLUP) [Support],"627,761.0000","627,761.0000",0.0000,0.0000,True
